# GNSS Relativistic Corrections Model
### Based on: Ayso & Kahveci (2025) *Meas. Sci. Technol.* 36 066318
**"Impacts of relativistic effects on GNSS signal path and precise point positioning"**

This notebook implements and analyses three relativistic corrections for GNSS signals:
1. **ERC** – Earth's Rotation Correction (Sagnac Effect) — *Paper Equations 7–9, 16*
2. **RCC** – Relativistic Clock Correction — *Paper Equations 11–12*
3. **RPRC** – Relativistic Path Range Correction (Shapiro Delay) — *Paper Equation 13*

**Dataset:** IISC IGS Station, DOY 242 / 2024 (29 August 2024)
- `IISC00IND_R_20242420000_01D_30S_MO.rnx` — RINEX 3 observation file (30-s GPS)
- `IGS0OPSFIN_20242420000_01D_15M_ORB.SP3` — IGS Final precise orbits (15-min)
- `iisc2420.24n` — RINEX broadcast navigation file

In [ ]:
# Install required GNSS libraries
!pip install -q georinex hatanaka

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import georinex as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.interpolate import BarycentricInterpolator

# ── Physical Constants (Paper Section 2) ──────────────────────────────────────
c       = 299_792_458.0          # Speed of light in vacuum [m/s]
omega_E = 7.2921151467e-5        # Earth rotation rate [rad/s]
mu      = 3.986_004_418e14       # Earth gravitational constant [m³/s²]

# GPS signal frequencies
f1 = 1_575.42e6                  # L1 frequency [Hz]
f2 = 1_227.60e6                  # L2 frequency [Hz]

print("Physical constants loaded.")
print(f"  c       = {c:.0f} m/s")
print(f"  omega_E = {omega_E} rad/s")
print(f"  mu      = {mu:.6e} m³/s²")

In [ ]:
# ── 1. Load RINEX Observation File ────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print("Loading optimized observation data... This should take less than a minute.")

obs_data = gr.load('IISC00IND_R_20242420000_01D_30S_MO.rnx', use='G')

print("Load complete!")

# Convert the original xarray dataset to a pandas DataFrame and drop empty rows
obs_df     = obs_data.to_dataframe().dropna(how='all')
obs_df_null = obs_data.to_dataframe()

obs_df.info()
obs_df.head()

In [ ]:
# ── 2. Load SP3 Precise Ephemeris File ────────────────────────────────────────
sp3_data = gr.load('IGS0OPSFIN_20242420000_01D_15M_ORB.SP3')

# Convert the original xarray dataset to a pandas DataFrame and drop empty rows
sp3_df      = sp3_data.to_dataframe().dropna(how='all')
sp3_df_null = sp3_data.to_dataframe()

sp3_df.info()
sp3_df.head()

In [ ]:
# ── 3. Load RINEX Navigation File (Broadcast Ephemeris) ───────────────────────
nav_data = gr.load('iisc2420.24n')

# Convert to a pandas DataFrame and drop rows with missing values
df_nav      = nav_data.to_dataframe().dropna(how='all')
df_nav_null = nav_data.to_dataframe()

df_nav.info()
df_nav.head()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DATA PREPROCESSING
# ════════════════════════════════════════════════════════════════════════════════

# ── 3.1  SP3: Reshape from (time, sv, ECEF) → (time, sv) with columns x, y, z ──
# Positions in SP3 are in km; convert to metres
sp3_pos = sp3_df['position'].unstack(level='ECEF') * 1000.0   # km → m
sp3_pos.index.names = ['time', 'sv']
sp3_pos.columns.name = None

# SP3 satellite clock offset: µs → seconds  (999999.999999 = invalid)
sp3_clk = (sp3_df['clock']
           .unstack(level='ECEF')
           .iloc[:, 0]              # same value repeated for x/y/z
           * 1e-6)                  # µs → s
sp3_clk = sp3_clk[sp3_clk < 900.0]  # remove invalid entries
sp3_clk.index.names = ['time', 'sv']

print("SP3 positions (m) – first 3 rows:")
print(sp3_pos.head(3))
print(f"\nSP3 epochs  : {sp3_pos.index.get_level_values('time').nunique()}")
print(f"SP3 sats    : {sp3_pos.index.get_level_values('sv').nunique()}")

# ── 3.2  Observations: derive P1, P2 and ionosphere-free combination ───────────
obs_work = obs_df.copy()
obs_work['P1'] = obs_work['C1W'].fillna(obs_work['C1C'])   # L1 pseudorange [m]
obs_work['P2'] = obs_work['C2W']                           # L2 pseudorange [m]
obs_work = obs_work.dropna(subset=['P1', 'P2'])

# Ionosphere-Free combination (Paper Eq. 14)
# P_IF = (f1² · P1 − f2² · P2) / (f1² − f2²)
f1sq = f1 ** 2
f2sq = f2 ** 2
obs_work['P_IF'] = (f1sq * obs_work['P1'] - f2sq * obs_work['P2']) / (f1sq - f2sq)

print(f"\nDual-frequency observations : {len(obs_work)}")
print(f"P_IF range : {obs_work['P_IF'].min():.0f} – {obs_work['P_IF'].max():.0f} m")

# ── 3.3  Receiver approximate position (ECEF, metres) ─────────────────────────
try:
    hdr   = gr.rinexheader('IISC00IND_R_20242420000_01D_30S_MO.rnx')
    p_rec = np.array(hdr['position'])
    print(f"\nReceiver position (from RINEX header) : {p_rec} m")
except Exception:
    # Fallback: published IISC IGS coordinates
    p_rec = np.array([1_633_441.0, 5_932_731.0, 1_425_274.0])
    print(f"\nReceiver position (IISC fallback)     : {p_rec} m")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SP3 LAGRANGE INTERPOLATION  (standard GNSS practice, 9 points)
# ════════════════════════════════════════════════════════════════════════════════

# Reference epoch for this day
DAY0 = pd.Timestamp('2024-08-29 00:00:00')

# Pre-build per-satellite look-up tables once for speed
print("Pre-building SP3 interpolation tables...")
_sp3_cache = {}
for sv, grp in sp3_pos.groupby(level='sv'):
    t_arr = np.array([(t - DAY0).total_seconds()
                      for t in grp.index.get_level_values('time')])
    _sp3_cache[sv] = {
        't': t_arr,
        'x': grp['x'].values,
        'y': grp['y'].values,
        'z': grp['z'].values,
    }
print(f"  Cached {len(_sp3_cache)} satellites.")


def interp_sp3(sv: str, t_sec: float, n_pts: int = 9) -> np.ndarray | None:
    """
    Barycentric Lagrange interpolation of SP3 satellite position.

    Parameters
    ----------
    sv    : satellite ID, e.g. 'G05'
    t_sec : seconds from 2024-08-29 00:00:00 UTC
    n_pts : number of interpolation nodes (default 9 – GNSS standard)

    Returns
    -------
    np.ndarray([x, y, z]) in metres, or None if data unavailable.
    """
    if sv not in _sp3_cache:
        return None
    d   = _sp3_cache[sv]
    idx = int(np.searchsorted(d['t'], t_sec))
    half = n_pts // 2
    i0   = max(0, min(idx - half, len(d['t']) - n_pts))
    i1   = i0 + n_pts
    tk   = d['t'][i0:i1]
    if len(tk) < 3:
        return None
    try:
        x = float(BarycentricInterpolator(tk, d['x'][i0:i1])(t_sec))
        y = float(BarycentricInterpolator(tk, d['y'][i0:i1])(t_sec))
        z = float(BarycentricInterpolator(tk, d['z'][i0:i1])(t_sec))
        return np.array([x, y, z])
    except Exception:
        return None


def sat_velocity_sp3(sv: str, t_sec: float, edt: float = 1.0) -> np.ndarray | None:
    """
    Satellite velocity via numerical differentiation of SP3 positions.
    Paper Equation 6:  ṽ = (p̃(t) − p̃(t − Δt)) / Δt
    """
    p0 = interp_sp3(sv, t_sec - edt)
    p1 = interp_sp3(sv, t_sec)
    if p0 is None or p1 is None:
        return None
    return (p1 - p0) / edt   # m/s


print("Interpolation functions ready.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# RELATIVISTIC CORRECTION FUNCTIONS
# Reference: Ayso & Kahveci (2025) Meas. Sci. Technol. 36 066318
# ════════════════════════════════════════════════════════════════════════════════


def signal_transmission_time(t_SRT_sec: float, sv: str,
                             p_rec: np.ndarray, max_iter: int = 5):
    """
    Iteratively compute Signal Transmission Time (STT).
    Paper Equations 1–3:
        P  = c · (t_SRT − t_STT)
        dt = P / c
    Returns (t_STT_sec, p_sat_at_STT, travel_time) or (None, None, None).
    """
    dt = 0.075                          # initial guess: ~75 ms
    for _ in range(max_iter):
        t_STT = t_SRT_sec - dt
        p_sat = interp_sp3(sv, t_STT)
        if p_sat is None:
            return None, None, None
        dist   = np.linalg.norm(p_sat - p_rec)
        dt_new = dist / c
        if abs(dt_new - dt) < 1e-12:
            break
        dt = dt_new
    return t_STT, p_sat, dt


def apply_ERC(p_sat_STT: np.ndarray, v_sat_STT,
              p_rec: np.ndarray):
    """
    Earth Rotation Correction — Sagnac Effect.
    Paper Equations 7–9.

    Rotates satellite position/velocity from the signal-transmission-time
    ECEF frame to the signal-reception-time ECEF frame using R3[θ]:

        R3[θ] = | cos θ   sin θ   0 |
                |−sin θ   cos θ   0 |
                |  0        0     1 |

        θ = ωE · |p̃_STT − p_rec| / c
    """
    dist  = np.linalg.norm(p_sat_STT - p_rec)
    theta = omega_E * dist / c           # rotation angle [rad]
    ct, st = np.cos(theta), np.sin(theta)

    # Rotate position (Paper Eq. 7)
    p_SRT = np.array([
         ct * p_sat_STT[0] + st * p_sat_STT[1],
        -st * p_sat_STT[0] + ct * p_sat_STT[1],
             p_sat_STT[2]
    ])

    # Rotate velocity (Paper Eq. 8)
    if v_sat_STT is not None:
        v_SRT = np.array([
             ct * v_sat_STT[0] + st * v_sat_STT[1],
            -st * v_sat_STT[0] + ct * v_sat_STT[1],
                 v_sat_STT[2]
        ])
    else:
        v_SRT = None

    return p_SRT, v_SRT


def erc_range(p_sat_STT: np.ndarray, p_sat_SRT: np.ndarray,
              p_rec: np.ndarray):
    """
    ERC effect on signal-path (pseudorange).
    Paper Equation 16:
        ERC = ρ_uncorrected − ρ_corrected
            = |p̃_STT − p_rec| − |p_SRT − p_rec|
    Returns (erc_metres, rho_corrected_metres).
    """
    rho_u = np.linalg.norm(p_sat_STT - p_rec)
    rho_c = np.linalg.norm(p_sat_SRT - p_rec)
    return rho_u - rho_c, rho_c


def rcc_precise(p_sat_SRT: np.ndarray, v_sat_SRT) -> float:
    """
    Relativistic Clock Correction — precise ephemeris method.
    Paper Equation 12:
        δt^s_rel = −2 / c² · (p_SRT · v_SRT)   [seconds]
    Returns correction in metres (·c).
    """
    if v_sat_SRT is None:
        return 0.0
    return (-2.0 / c**2) * float(np.dot(p_sat_SRT, v_sat_SRT)) * c   # m


def _kepler(M: float, e: float) -> float:
    """Solve Kepler's equation M = E − e·sin E via Newton–Raphson."""
    E = M
    for _ in range(50):
        dE = (M - E + e * np.sin(E)) / (1.0 - e * np.cos(E))
        E += dE
        if abs(dE) < 1e-13:
            break
    return E


def rcc_broadcast(sv: str, t_STT_sec: float, df_nav) -> float:
    """
    Relativistic Clock Correction — broadcast ephemeris method.
    Paper Equation 11:
        δt^s_rel = −2/c² · √(a · µ) · e · sin E   [seconds]
    Returns correction in metres (·c).
    """
    if sv not in df_nav.index.get_level_values('sv'):
        return 0.0
    nav_sv = df_nav.xs(sv, level='sv')

    # Select ephemeris whose Toe is closest to t_STT within the day
    toes   = nav_sv['Toe'].values
    t_day  = t_STT_sec % 86400.0
    best   = nav_sv.iloc[int(np.argmin(np.abs(toes - t_day)))]

    a   = best['sqrtA'] ** 2
    e   = best['Eccentricity']
    M0  = best['M0']
    dn  = best['DeltaN']
    Toe = best['Toe']

    n  = np.sqrt(mu / a**3) + dn
    tk = t_STT_sec % 86400.0 - Toe
    if tk >  302400: tk -= 604800
    if tk < -302400: tk += 604800

    Ek      = _kepler(M0 + n * tk, e)
    dts_rel = (-2.0 / c**2) * np.sqrt(a * mu) * e * np.sin(Ek)  # s
    return dts_rel * c   # m


def rprc(p_sat_SRT: np.ndarray, p_rec: np.ndarray) -> float:
    """
    Relativistic Path Range Correction (Shapiro delay).
    Paper Equation 13:
        Δr_path = 2µ/c³ · ln[ (|p_sat| + |p_rec| + |p_sat_rec|) /
                               (|p_sat| + |p_rec| − |p_sat_rec|) ]
    Returns correction in metres.
    """
    n_sat     = np.linalg.norm(p_sat_SRT)
    n_rec     = np.linalg.norm(p_rec)
    n_sat_rec = np.linalg.norm(p_sat_SRT - p_rec)
    den = n_sat + n_rec - n_sat_rec
    if den <= 0.0:
        return 0.0
    return (2.0 * mu / c**3) * np.log((n_sat + n_rec + n_sat_rec) / den) * c  # m


print("All relativistic correction functions defined.")
print("  ✔ signal_transmission_time()  — Paper Eq. 1–3")
print("  ✔ apply_ERC()                 — Paper Eq. 7–9")
print("  ✔ erc_range()                 — Paper Eq. 16")
print("  ✔ rcc_precise()               — Paper Eq. 12")
print("  ✔ rcc_broadcast()             — Paper Eq. 11")
print("  ✔ rprc()                      — Paper Eq. 13")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MAIN COMPUTATION — apply all three relativistic corrections to every obs epoch
# ════════════════════════════════════════════════════════════════════════════════

print(f"Processing {len(obs_work)} satellite observations …\n")

rows = []

for i, (idx, row) in enumerate(obs_work.iterrows()):
    if i % 3000 == 0:
        pct = 100 * i / len(obs_work)
        print(f"  {i:>6}/{len(obs_work)}  ({pct:.0f}%) …")

    t_SRT   = idx[0]                                    # pandas Timestamp
    sv      = idx[1]                                    # e.g. 'G05'
    t_SRT_s = (t_SRT - DAY0).total_seconds()

    # ── Step 1: Signal Transmission Time (Paper Eq. 1–3) ──────────────────────
    t_STT_s, p_sat_STT, dt_flight = signal_transmission_time(t_SRT_s, sv, p_rec)
    if p_sat_STT is None:
        continue

    # ── Step 2: Satellite velocity at STT (Paper Eq. 6) ──────────────────────
    v_sat_STT = sat_velocity_sp3(sv, t_STT_s)

    # ── Step 3: ERC — Earth Rotation Correction (Paper Eq. 7–9) ─────────────
    p_sat_SRT, v_sat_SRT = apply_ERC(p_sat_STT, v_sat_STT, p_rec)

    # ── Step 4: ERC effect on pseudorange (Paper Eq. 16) ────────────────────
    erc_m, rho_corr = erc_range(p_sat_STT, p_sat_SRT, p_rec)

    # ── Step 5: RCC — precise-ephemeris method (Paper Eq. 12) ───────────────
    rcc_pr_m = rcc_precise(p_sat_SRT, v_sat_SRT)

    # ── Step 6: RCC — broadcast-ephemeris method (Paper Eq. 11) ─────────────
    rcc_bc_m = rcc_broadcast(sv, t_STT_s, df_nav)

    # ── Step 7: RPRC — Shapiro delay (Paper Eq. 13) ──────────────────────────
    rprc_m = rprc(p_sat_SRT, p_rec)

    # ── Step 8: Corrected pseudorange ────────────────────────────────────────
    # P_IF_corrected = P_IF − c·(δt_rcc) − Δr_path   [Paper Eq. 14]
    P_IF_corr = row['P_IF'] - rcc_pr_m - rprc_m

    rho_uncorr = np.linalg.norm(p_sat_STT - p_rec)

    rows.append(dict(
        time          = t_SRT,
        sv            = sv,
        P1            = row['P1'],
        P2            = row['P2'],
        P_IF          = row['P_IF'],
        rho_uncorr    = rho_uncorr,
        rho_ERC       = rho_corr,
        ERC_m         = erc_m,
        RCC_precise_m = rcc_pr_m,
        RCC_bcast_m   = rcc_bc_m,
        RPRC_m        = rprc_m,
        P_IF_corr     = P_IF_corr,
        dt_flight     = dt_flight,
    ))

res = pd.DataFrame(rows).set_index(['time', 'sv'])
print(f"\nDone. {len(res)} observations processed successfully.")
res.head()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STATISTICAL SUMMARY — matches paper Tables 2, 5, 8 (station: IISC)
# ════════════════════════════════════════════════════════════════════════════════

def stats(series, label):
    """Return a one-row summary dict, using absolute values for ERC/RCC/RPRC."""
    a = series.abs()
    return dict(
        Correction = label,
        Max_m  = round(a.max(), 4),
        Min_m  = round(a.min(), 4),
        Mean_m = round(a.mean(), 4),
        Std_m  = round(series.std(), 4),
    )

summary = pd.DataFrame([
    stats(res['ERC_m'],         'ERC  (Eq. 16)'),
    stats(res['RCC_precise_m'], 'RCC  (Eq. 12) — precise'),
    stats(res['RCC_bcast_m'],   'RCC  (Eq. 11) — broadcast'),
    stats(res['RPRC_m'],        'RPRC (Eq. 13) — Shapiro'),
])
summary = summary.set_index('Correction')

print("=" * 65)
print("  Relativistic corrections on GNSS signal path — IISC station")
print("  (Ref: Ayso & Kahveci 2025, Tables 2, 5, 8)")
print("=" * 65)
print(summary.to_string())
print("=" * 65)

# Expected order-of-magnitude from the paper:
#   ERC  : mean ~14–17 m,   max ~24–38 m
#   RCC  : mean  ~3.7–3.9 m, max ~17 m
#   RPRC : mean  ~0.014–0.015 m

# Per-satellite summary
print("\nPer-satellite ERC statistics (mean |ERC| in metres):")
sv_erc = (res['ERC_m'].abs()
            .groupby(level='sv')
            .agg(['max', 'mean', 'min'])
            .round(3)
            .sort_values('mean', ascending=False))
print(sv_erc.to_string())

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VISUALISATION  1 — ERC effect on signal path  (Paper Figure 8)
# ════════════════════════════════════════════════════════════════════════════════

erc_time = res['ERC_m'].reset_index()
erc_time['t_hr'] = (erc_time['time'] - DAY0).dt.total_seconds() / 3600.0

fig, ax = plt.subplots(figsize=(14, 5))
for sv_id, grp in erc_time.groupby('sv'):
    ax.plot(grp['t_hr'], grp['ERC_m'], linewidth=0.8, alpha=0.7, label=sv_id)

ax.set_xlabel('Time [hours from 00:00 UTC, 2024-08-29]')
ax.set_ylabel('ERC effect on signal path [m]')
ax.set_title('Earth Rotation Correction (ERC) Effect on GNSS Signal Path\n'
             'IISC Station | Ayso & Kahveci (2025) Eq. 16 — cf. Figure 8')
ax.grid(True, alpha=0.3)
ax.legend(ncol=6, fontsize=7, loc='upper right')
mean_abs = res['ERC_m'].abs().mean()
ax.text(0.01, 0.97,
        f"Mean |ERC| = {mean_abs:.2f} m  |  Max |ERC| = {res['ERC_m'].abs().max():.2f} m",
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
plt.tight_layout()
plt.savefig('fig1_ERC_signal_path.png', dpi=150)
plt.show()
print("Figure saved: fig1_ERC_signal_path.png")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VISUALISATION  2 — RCC effect on signal path  (Paper Figure 10)
# ════════════════════════════════════════════════════════════════════════════════

rcc_time = res[['RCC_precise_m', 'RCC_bcast_m']].reset_index()
rcc_time['t_hr'] = (rcc_time['time'] - DAY0).dt.total_seconds() / 3600.0

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=False)

# ── Left: RCC precise ephemeris (Eq. 12) ──────────────────────────────────────
for sv_id, grp in rcc_time.groupby('sv'):
    axes[0].plot(grp['t_hr'], grp['RCC_precise_m'].abs(),
                 linewidth=0.8, alpha=0.7)
axes[0].set_xlabel('Time [hours]')
axes[0].set_ylabel('|RCC| [m]')
axes[0].set_title('RCC — Precise Ephemeris (Eq. 12)\ncf. Paper Figure 10')
axes[0].grid(True, alpha=0.3)
m = res['RCC_precise_m'].abs().mean()
axes[0].axhline(m, color='red', linestyle='--', label=f'Mean = {m:.3f} m')
axes[0].legend()

# ── Right: comparison precise vs broadcast ─────────────────────────────────────
axes[1].scatter(res['RCC_precise_m'].abs().values,
                res['RCC_bcast_m'].abs().values,
                s=3, alpha=0.3, color='steelblue')
lim = max(res['RCC_precise_m'].abs().max(), res['RCC_bcast_m'].abs().max())
axes[1].plot([0, lim], [0, lim], 'r--', label='1:1 line')
axes[1].set_xlabel('|RCC| precise [m]')
axes[1].set_ylabel('|RCC| broadcast [m]')
axes[1].set_title('RCC: Precise (Eq. 12) vs Broadcast (Eq. 11)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig('fig2_RCC_signal_path.png', dpi=150)
plt.show()
print("Figure saved: fig2_RCC_signal_path.png")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VISUALISATION  3 — RPRC effect on signal path  (Paper Figure 12)
# ════════════════════════════════════════════════════════════════════════════════

rprc_time = res['RPRC_m'].reset_index()
rprc_time['t_hr'] = (rprc_time['time'] - DAY0).dt.total_seconds() / 3600.0

fig, ax = plt.subplots(figsize=(14, 5))
for sv_id, grp in rprc_time.groupby('sv'):
    ax.plot(grp['t_hr'], grp['RPRC_m'] * 100,   # convert m → cm
            linewidth=0.9, alpha=0.7, label=sv_id)

ax.set_xlabel('Time [hours from 00:00 UTC, 2024-08-29]')
ax.set_ylabel('RPRC effect on signal path [cm]')
ax.set_title('Relativistic Path Range Correction (RPRC / Shapiro Delay) Effect\n'
             'IISC Station | Ayso & Kahveci (2025) Eq. 13 — cf. Figure 12')
ax.grid(True, alpha=0.3)
ax.legend(ncol=6, fontsize=7, loc='lower right')
m = res['RPRC_m'].mean() * 100
ax.text(0.01, 0.97,
        f"Mean RPRC = {m:.4f} cm  |  Max RPRC = {res['RPRC_m'].max()*100:.4f} cm",
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
plt.tight_layout()
plt.savefig('fig3_RPRC_signal_path.png', dpi=150)
plt.show()
print("Figure saved: fig3_RPRC_signal_path.png")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VISUALISATION  4 — Pseudorange before / after corrections
# ════════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('GNSS Pseudorange: Relativistic Corrections Applied\nIISC Station | Ayso & Kahveci (2025)', fontsize=13)

res_r = res.reset_index()
res_r['t_hr'] = (res_r['time'] - DAY0).dt.total_seconds() / 3600.0

# ── Top-left: Uncorrected vs corrected P_IF for one representative satellite ──
example_sv = res_r['sv'].value_counts().index[0]   # most observed satellite
sub = res_r[res_r['sv'] == example_sv]
axes[0, 0].plot(sub['t_hr'], sub['P_IF'],      'gray',     lw=1,   label='P_IF uncorrected')
axes[0, 0].plot(sub['t_hr'], sub['P_IF_corr'], 'tab:blue', lw=1.5, label='P_IF corrected (RCC+RPRC)')
axes[0, 0].set_title(f'P_IF: satellite {example_sv}')
axes[0, 0].set_xlabel('Time [h]'); axes[0, 0].set_ylabel('Pseudorange [m]')
axes[0, 0].legend(fontsize=8); axes[0, 0].grid(True, alpha=0.3)

# ── Top-right: Total correction magnitude over time ───────────────────────────
res_r['total_corr'] = res_r['ERC_m'] + res_r['RCC_precise_m'] + res_r['RPRC_m']
for sv_id, grp in res_r.groupby('sv'):
    axes[0, 1].plot(grp['t_hr'], grp['total_corr'].abs(),
                    linewidth=0.7, alpha=0.6)
axes[0, 1].set_title('Total |ERC + RCC + RPRC| per satellite')
axes[0, 1].set_xlabel('Time [h]'); axes[0, 1].set_ylabel('Total correction [m]')
axes[0, 1].grid(True, alpha=0.3)

# ── Bottom-left: Correction breakdown — box plots ─────────────────────────────
corr_data = pd.DataFrame({
    'ERC': res['ERC_m'].abs().values,
    'RCC (precise)': res['RCC_precise_m'].abs().values,
    'RPRC (x100)': res['RPRC_m'].abs().values * 100,
})
corr_data.plot.box(ax=axes[1, 0], notch=False, showfliers=False)
axes[1, 0].set_title('Distribution of Correction Magnitudes\n(RPRC scaled x100 for visibility)')
axes[1, 0].set_ylabel('Correction [m]')
axes[1, 0].grid(True, alpha=0.3)

# ── Bottom-right: Signal travel time distribution ─────────────────────────────
axes[1, 1].hist(res['dt_flight'].values * 1000, bins=60,
                color='steelblue', edgecolor='white', linewidth=0.3)
axes[1, 1].set_title('Signal Flight Time Distribution')
axes[1, 1].set_xlabel('Flight time [ms]'); axes[1, 1].set_ylabel('Count')
axes[1, 1].grid(True, alpha=0.3)
mean_dt = res['dt_flight'].mean() * 1000
axes[1, 1].axvline(mean_dt, color='red', linestyle='--',
                   label=f'Mean = {mean_dt:.1f} ms')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('fig4_pseudorange_corrections.png', dpi=150)
plt.show()
print("Figure saved: fig4_pseudorange_corrections.png")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# VISUALISATION  5 — ERC-induced shift in satellite X/Y coordinates
#                    (Paper Figures 4 & 5)
# ════════════════════════════════════════════════════════════════════════════════

print("Computing ERC-induced satellite coordinate shifts …")

coord_rows = []
for i, (idx, row) in enumerate(obs_work.iterrows()):
    t_SRT   = idx[0]
    sv      = idx[1]
    t_SRT_s = (t_SRT - DAY0).total_seconds()

    t_STT_s, p_STT, _ = signal_transmission_time(t_SRT_s, sv, p_rec)
    if p_STT is None:
        continue
    v_STT = sat_velocity_sp3(sv, t_STT_s)
    p_SRT, v_SRT = apply_ERC(p_STT, v_STT, p_rec)

    dX = p_STT[0] - p_SRT[0]
    dY = p_STT[1] - p_SRT[1]
    # dZ = 0 by definition (Paper Eqs. 19, 22)

    coord_rows.append(dict(time=t_SRT, sv=sv,
                           dX=dX, dY=dY,
                           t_hr=(t_SRT - DAY0).total_seconds() / 3600))

coord_df = pd.DataFrame(coord_rows)
print(f"Done. {len(coord_df)} records.")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('ERC-Induced Satellite Coordinate Shifts (dX, dY)\n'
             'IISC Station | Ayso & Kahveci (2025) Eq. 17–18 — cf. Figures 4 & 5',
             fontsize=12)

for sv_id, grp in coord_df.groupby('sv'):
    axes[0].plot(grp['t_hr'], grp['dX'], linewidth=0.8, alpha=0.7, label=sv_id)
    axes[1].plot(grp['t_hr'], grp['dY'], linewidth=0.8, alpha=0.7, label=sv_id)

for ax, lbl in zip(axes, ['dX  [m]', 'dY  [m]']):
    ax.set_xlabel('Time [hours]')
    ax.set_ylabel(lbl)
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=5, fontsize=7)

axes[0].set_title('X-axis coordinate shift due to ERC')
axes[1].set_title('Y-axis coordinate shift due to ERC')

plt.tight_layout()
plt.savefig('fig5_ERC_coordinate_shift.png', dpi=150)
plt.show()
print("Figure saved: fig5_ERC_coordinate_shift.png")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# FINAL MODEL SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

print("=" * 68)
print("  GNSS RELATIVISTIC CORRECTIONS MODEL — FINAL SUMMARY")
print("  Station : IISC (Indian Institute of Science, Bangalore)")
print("  Date    : 2024-08-29  (DOY 242)")
print("  Ref     : Ayso & Kahveci (2025) Meas. Sci. Technol. 36 066318")
print("=" * 68)

corrections = {
    'ERC  (Earth Rotation, Sagnac) — Eq. 16' : res['ERC_m'],
    'RCC  (Rel. Clock, precise)    — Eq. 12' : res['RCC_precise_m'],
    'RCC  (Rel. Clock, broadcast)  — Eq. 11' : res['RCC_bcast_m'],
    'RPRC (Shapiro delay)          — Eq. 13' : res['RPRC_m'],
}

for label, series in corrections.items():
    a = series.abs()
    print(f"\n  {label}")
    print(f"    Max   = {a.max():>10.4f} m")
    print(f"    Mean  = {a.mean():>10.4f} m")
    print(f"    Min   = {a.min():>10.4f} m")
    print(f"    Std   = {series.std():>10.4f} m")

print("\n" + "-" * 68)
print("  Comparison with paper (different station — values should be similar):")
print("    ERC  : paper mean ~14–17 m, max ~24–38 m")
print("    RCC  : paper mean  ~3.7–3.9 m, max ~17 m")
print("    RPRC : paper mean  ~0.014–0.015 m")
print("=" * 68)